# MandiDog IA — Entrenamiento en Google Colab

Clasificador binario (`con_signos` / `sin_signos`) con transfer learning sobre MobileNetV2, equivalente a `ml/train.py`.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU**.

> Los resultados del modelo son orientativos y no reemplazan el diagnóstico de un médico veterinario.

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

## 1. Datos

Suba a Google Drive la carpeta `mandidog/data` con esta estructura (vea `ml/README.md` para obtener las imágenes):

```
mandidog/data/
├── con_signos/
└── sin_signos/
```

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
CARPETA_DATOS = "/content/drive/MyDrive/mandidog/data"

**Opcional:** descargar el dataset público *dog dental* de Roboflow Universe (CC BY 4.0). Después hay que organizar las imágenes en `con_signos/` y `sin_signos/` y apuntar `CARPETA_DATOS` a esa carpeta.

In [ ]:
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="SU_CLAVE")
# dataset = rf.workspace("moriahai").project("dog-dental").version(1).download("folder")

## 2. Código de entrenamiento (idéntico a `ml/train.py`)

In [ ]:
from __future__ import annotations

import argparse
import json
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorflow as tf

TAMANO_IMAGEN = (224, 224)
CLASES = ["sin_signos", "con_signos"]  # índice 1 = "con signos": la salida sigmoide es su probabilidad
EXTENSIONES = {".jpg", ".jpeg", ".png"}


def argumentos() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Entrena el modelo de MandiDog IA")
    parser.add_argument("--data", type=Path, default=Path("data"), help="Carpeta con con_signos/ y sin_signos/")
    parser.add_argument("--salida", type=Path, default=Path("models/mandidog_v1.keras"))
    parser.add_argument("--metricas", type=Path, default=Path("metrics.json"))
    parser.add_argument("--epocas-base", type=int, default=15)
    parser.add_argument("--epocas-ajuste", type=int, default=15)
    parser.add_argument("--lote", type=int, default=16)
    parser.add_argument("--capas-ajuste", type=int, default=30, help="Capas finales de la base a descongelar")
    parser.add_argument("--prueba", type=float, default=0.15, help="Proporción del conjunto de prueba")
    parser.add_argument("--validacion", type=float, default=0.15, help="Proporción del conjunto de validación")
    parser.add_argument("--semilla", type=int, default=42)
    return parser.parse_args()


def fijar_semillas(semilla: int) -> None:
    random.seed(semilla)
    np.random.seed(semilla)
    tf.random.set_seed(semilla)


def listar_imagenes(carpeta_datos: Path) -> tuple[list[str], list[int]]:
    rutas, etiquetas = [], []
    for indice, clase in enumerate(CLASES):
        carpeta = carpeta_datos / clase
        if not carpeta.is_dir():
            raise SystemExit(f"No existe la carpeta {carpeta}. Revise ml/README.md para preparar los datos.")
        archivos = sorted(p for p in carpeta.rglob("*") if p.suffix.lower() in EXTENSIONES)
        if not archivos:
            raise SystemExit(f"La carpeta {carpeta} no contiene imágenes JPG o PNG.")
        rutas += [str(p) for p in archivos]
        etiquetas += [indice] * len(archivos)
        print(f"  {clase}: {len(archivos)} imágenes")
    return rutas, etiquetas


def dividir_estratificado(rutas, etiquetas, prop_prueba, prop_val, semilla):
    """Divide en entrenamiento, validación y prueba conservando la proporción de cada clase."""
    generador = np.random.default_rng(semilla)
    conjuntos = {"entrenamiento": ([], []), "validacion": ([], []), "prueba": ([], [])}
    etiquetas_np = np.array(etiquetas)
    for clase in np.unique(etiquetas_np):
        indices = np.where(etiquetas_np == clase)[0]
        generador.shuffle(indices)
        n_prueba = max(1, int(round(len(indices) * prop_prueba)))
        n_val = max(1, int(round(len(indices) * prop_val)))
        cortes = {
            "prueba": indices[:n_prueba],
            "validacion": indices[n_prueba : n_prueba + n_val],
            "entrenamiento": indices[n_prueba + n_val :],
        }
        for nombre, idx in cortes.items():
            conjuntos[nombre][0].extend(rutas[i] for i in idx)
            conjuntos[nombre][1].extend(int(etiquetas[i]) for i in idx)
    return conjuntos


def cargar_imagen(ruta: tf.Tensor, etiqueta: tf.Tensor):
    """Lee, decodifica y normaliza a [0, 1]. Debe coincidir con el preprocesamiento de la API."""
    contenido = tf.io.read_file(ruta)
    imagen = tf.io.decode_image(contenido, channels=3, expand_animations=False)
    imagen = tf.image.resize(imagen, TAMANO_IMAGEN, method="bilinear")
    imagen = tf.cast(imagen, tf.float32) / 255.0
    return imagen, tf.cast(etiqueta, tf.float32)


def crear_dataset(rutas, etiquetas, lote: int, mezclar: bool, semilla: int) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if mezclar:
        ds = ds.shuffle(len(rutas), seed=semilla, reshuffle_each_iteration=True)
    ds = ds.map(cargar_imagen, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(lote).prefetch(tf.data.AUTOTUNE)


def construir_modelo() -> tuple[tf.keras.Model, tf.keras.Model]:
    aumento = tf.keras.Sequential(
        [
            tf.keras.layers.RandomRotation(0.08),
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomBrightness(0.2, value_range=(0.0, 1.0)),
            tf.keras.layers.RandomContrast(0.2),
        ],
        name="aumento_datos",
    )
    base = tf.keras.applications.MobileNetV2(
        input_shape=TAMANO_IMAGEN + (3,), include_top=False, weights="imagenet"
    )
    base.trainable = False

    entrada = tf.keras.Input(shape=TAMANO_IMAGEN + (3,), name="imagen")
    x = aumento(entrada)
    x = tf.keras.layers.Rescaling(2.0, offset=-1.0, name="escala_mobilenet")(x)  # [0,1] -> [-1,1]
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    salida = tf.keras.layers.Dense(1, activation="sigmoid", name="prob_con_signos")(x)
    return tf.keras.Model(entrada, salida, name="mandidog"), base


def compilar(modelo: tf.keras.Model, tasa: float) -> None:
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=tasa),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )


def pesos_de_clase(etiquetas: list[int]) -> dict[int, float]:
    total = len(etiquetas)
    conteo = np.bincount(etiquetas, minlength=2)
    return {i: total / (2.0 * max(1, conteo[i])) for i in range(2)}


def calcular_metricas(y_real: np.ndarray, y_prob: np.ndarray, umbral: float = 0.5) -> dict:
    y_pred = (y_prob >= umbral).astype(int)
    vp = int(np.sum((y_pred == 1) & (y_real == 1)))
    vn = int(np.sum((y_pred == 0) & (y_real == 0)))
    fp = int(np.sum((y_pred == 1) & (y_real == 0)))
    fn = int(np.sum((y_pred == 0) & (y_real == 1)))

    def dividir(a, b):
        return round(a / b, 4) if b else 0.0

    precision = dividir(vp, vp + fp)
    sensibilidad = dividir(vp, vp + fn)
    return {
        "umbral": umbral,
        "muestras_prueba": int(len(y_real)),
        "exactitud": dividir(vp + vn, len(y_real)),
        "precision": precision,
        "sensibilidad": sensibilidad,
        "especificidad": dividir(vn, vn + fp),
        "f1": dividir(2 * precision * sensibilidad, precision + sensibilidad),
        "matriz_confusion": {
            "orden": "filas = clase real, columnas = clase predicha, [sin_signos, con_signos]",
            "valores": [[vn, fp], [fn, vp]],
        },
    }


def entrenar(args: argparse.Namespace) -> dict:
    """Ejecuta todo el proceso y devuelve las métricas sobre el conjunto de prueba."""
    fijar_semillas(args.semilla)

    print("Leyendo imágenes...")
    rutas, etiquetas = listar_imagenes(args.data)
    conjuntos = dividir_estratificado(rutas, etiquetas, args.prueba, args.validacion, args.semilla)
    for nombre, (r, _) in conjuntos.items():
        print(f"  {nombre}: {len(r)} imágenes")

    ds_entrenamiento = crear_dataset(*conjuntos["entrenamiento"], args.lote, True, args.semilla)
    ds_validacion = crear_dataset(*conjuntos["validacion"], args.lote, False, args.semilla)
    ds_prueba = crear_dataset(*conjuntos["prueba"], args.lote, False, args.semilla)

    modelo, base = construir_modelo()
    pesos = pesos_de_clase(conjuntos["entrenamiento"][1])
    parada = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

    print("\nFase 1: base congelada")
    compilar(modelo, 1e-3)
    modelo.fit(ds_entrenamiento, validation_data=ds_validacion, epochs=args.epocas_base,
               class_weight=pesos, callbacks=[parada])

    print(f"\nFase 2: ajuste fino de las últimas {args.capas_ajuste} capas")
    base.trainable = True
    for capa in base.layers[: -args.capas_ajuste]:
        capa.trainable = False
    for capa in base.layers:
        if isinstance(capa, tf.keras.layers.BatchNormalization):
            capa.trainable = False  # mantiene estables las estadísticas de ImageNet
    compilar(modelo, 1e-5)
    modelo.fit(ds_entrenamiento, validation_data=ds_validacion, epochs=args.epocas_ajuste,
               class_weight=pesos, callbacks=[parada])

    print("\nEvaluando en el conjunto de prueba...")
    y_prob = modelo.predict(ds_prueba, verbose=0).reshape(-1)
    y_real = np.array(conjuntos["prueba"][1])
    metricas = calcular_metricas(y_real, y_prob)
    metricas.update(
        {
            "modelo": args.salida.stem,
            "arquitectura": "MobileNetV2 (ImageNet) + clasificador binario",
            "clases": CLASES,
            "fecha": datetime.now(timezone.utc).isoformat(),
            "tamano_conjuntos": {k: len(v[0]) for k, v in conjuntos.items()},
        }
    )

    args.salida.parent.mkdir(parents=True, exist_ok=True)
    modelo.save(args.salida)
    args.metricas.parent.mkdir(parents=True, exist_ok=True)
    args.metricas.write_text(json.dumps(metricas, indent=2, ensure_ascii=False), encoding="utf-8")

    print(json.dumps(metricas, indent=2, ensure_ascii=False))
    print(f"\nModelo guardado en {args.salida}")
    print(f"Métricas guardadas en {args.metricas}")
    return metricas

## 3. Entrenar y evaluar

In [ ]:
args = argparse.Namespace(
    data=Path(CARPETA_DATOS),
    salida=Path("models/mandidog_v1.keras"),
    metricas=Path("metrics.json"),
    epocas_base=15,
    epocas_ajuste=15,
    lote=16,
    capas_ajuste=30,
    prueba=0.15,
    validacion=0.15,
    semilla=42,
)
metricas = entrenar(args)

## 4. Descargar el modelo y las métricas

In [ ]:
from google.colab import files

files.download("models/mandidog_v1.keras")
files.download("metrics.json")

Copie `mandidog_v1.keras` a `api/models/` y reinicie la API para salir del modo demostración.